# NSW house prices — Part 2: hypotheses, models, and results

This notebook picks up after cleaning and feature engineering
(`notebooks/01_data_and_features.ipynb`) and covers the analysis:

1. **H1** — does a bigger house cost more? (pooled vs *within the same location*)
2. **H2** — is price per sqm lower for bigger houses? (and how much of that is an artefact)
3. **H3** — did the 2019 Sydney Metro opening raise prices near the stations?
4. **Models** — rolling-origin cross-validation and a final untouched holdout
5. **Feature value** — an ablation plus two falsification tests
6. The report itself

The questions are answered with the same code the report uses, so the numbers match.

> **This notebook writes nothing to disk.** It reads the tables the pipeline already
> wrote (`reports/tables/`), which is why it runs in about a minute rather than 45.
> The section at the end lists the commands that *do* produce them.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import json

from src.utils.config import REPORTS_DIR, OUT_CLEAN_PARQUET
from src.features.pipeline import OUT_FEATURES_PARQUET

TABLES = REPORTS_DIR / "tables"
FIGURES = REPORTS_DIR / "figures"

pd.set_option("display.width", 170)

frame = pd.read_parquet(OUT_FEATURES_PARQUET)
print(f"modelling frame: {len(frame):,} rows x {frame.shape[1]} columns")
print(f"tables available: {len(list(TABLES.glob('*.csv')))} CSVs")

### A note on samples and runtimes

The hypothesis functions below bootstrap and permute, which on all 1.7M rows takes
minutes. `pipelines/05_hypothesis_tests.py` handles this by capping the resampling at
800,000 rows (point estimates still use everything) and reports that in its output.

This notebook does the same thing at a smaller size, so it runs in well under a minute.
The **published numbers come from the committed tables**, which are produced by the
pipeline — not from these demo cells. Every result cell is labelled to make clear which
is which.

In [ ]:
# A fixed subsample for the resampling demos. Same idea as the pipeline's --max-rows.
SAMPLE_N = 120_000
SAMPLE_SEED = 36103
sample = frame.sample(n=min(SAMPLE_N, len(frame)), random_state=SAMPLE_SEED).copy()
print(f"demo sample: {len(sample):,} rows (of {len(frame):,})")

## 1. H1 — does a bigger house cost more?

The obvious move is to correlate `log(area)` with `log(price)`. Doing that gives a
**negative** number, which sounds absurd.

The reason is that this correlation mixes two different comparisons:

* **between locations** — bigger houses tend to be further from the city, where land is
  cheaper, so area correlates with *lower* prices
* **within a location** — comparing two houses in the same postcode, the bigger one
  costs more

The pooled correlation is dominated by the first. To separate them we add **fixed
effects**: give every postcode and every year its own intercept, so the estimate is
identified only from variation *within* a postcode in a given year.

### The three numbers

| measure | what it compares |
|---|---|
| Pearson on raw values | dollars, mixes everything |
| Pearson on logs (pooled) | proportional, still mixes locations |
| **within postcode + year** | the number we want |

In [ ]:
from src.evaluation.hypothesis_tests import bootstrap_correlation_ci

# On the demo sample. The published version runs this on 800k rows.
n_boot = 60
pearson = bootstrap_correlation_ci(sample, "area_sqm", "purchase_price", "pearson", False, n_boot)
spearman = bootstrap_correlation_ci(sample, "area_sqm", "purchase_price", "spearman", False, n_boot)
loglog = bootstrap_correlation_ci(sample, "area_sqm", "purchase_price", "pearson", True, n_boot)

print("DEMO SAMPLE:")
for name, res in (("Pearson", pearson), ("Spearman", spearman), ("Pearson on logs", loglog)):
    print(f"  {name:18s} {res['coefficient']:+.4f}   "
          f"95% CI [{res['ci_low']:+.4f}, {res['ci_high']:+.4f}]")
print()
print("PUBLISHED (reports/tables/h1_headline_correlations.csv):")
pd.read_csv(TABLES / "h1_headline_correlations.csv")

In [ ]:
from src.evaluation.hypothesis_tests import fe_regression

# Same question, but only using variation WITHIN a postcode and year.
within = fe_regression(sample, y="log_price", x="log_area",
                       absorb=["post_code", "year"], cluster_col="post_code",
                       label="within postcode + year")

print("DEMO SAMPLE:")
print(f"  pooled log-log correlation : {loglog['coefficient']:+.4f}")
print(f"  within postcode + year     : {within['coef']:+.4f}  "
      f"(se {within['se']:.4f}, n = {within['n']:,}, clusters = {within['n_clusters']:,})")
print()
print("The sign flips. The pooled negative number was a between-location artefact.")
print()
print("PUBLISHED (reports/tables/h1_within_location.csv):")
pd.read_csv(TABLES / "h1_within_location.csv")

In [ ]:
# The same test, done year by year: the positive within-location slope should be stable.
by_year = pd.read_csv(TABLES / "h1_by_year.csv")
by_year.tail(8)

### Reading it

If the within-location elasticity is about **+0.25**, it means: comparing houses in the
same postcode and year, a 10% larger house sells for roughly **2.5% more**.

That is much less than proportional (which would be 1.0). So area *does* raise the price,
but far less than one-for-one — which is the mirror image of H2.

## 2. H2 — is price per sqm lower for bigger houses?

The common claim is "bigger houses are cheaper per square metre", and the data seems to
agree loudly: `corr(log area, log price_per_sqm)` is about **−0.51**.

Here is the trap. Price per sqm is

```
price_per_sqm = price / area
```

**Area is in the denominator.** Even if price had nothing to do with area, this ratio
would still produce a negative correlation with area — purely mechanically.

### The test: a shuffled-price placebo

Keep the areas exactly as they are, but **randomly shuffle the prices** across
transactions. That destroys every genuine economic relationship while preserving the
mechanical structure `price/area`.

Whatever correlation survives is the artefact. Whatever the real correlation exceeds it
by is the economic effect.

In [ ]:
from src.evaluation.hypothesis_tests import test_h2_unit_price

# 40 shuffled-price placebo draws on the demo sample. The published run uses 200 on 800k.
h2 = test_h2_unit_price(sample, n_placebo=40)

observed = h2.observed["coefficient"]
placebo_median = float(h2.placebo["coefficient"].median())
mechanical_share = placebo_median / observed

print("DEMO SAMPLE:")
print(f"  observed  corr(log area, log unit price) : {observed:+.4f}  (n = {h2.observed['n']:,})")
print(f"  placebo median (prices shuffled at random): {placebo_median:+.4f}")
print(f"  share of the observed value that is mechanical: {mechanical_share:.1%}")
print()
print("Most of the famous negative relationship is the ratio's denominator,")
print("not evidence that buyers get a discount for size.")
print()
print("PUBLISHED (reports/tables/h2_ratio_correlation.csv):")
print(pd.read_csv(TABLES / "h2_ratio_correlation.csv").to_string(index=False))

In [ ]:
print("PUBLISHED interpretation (reports/tables/hypothesis_summary.json):")
summary = json.loads((TABLES / "hypothesis_summary.json").read_text())
print()
print("H2:", summary["h2"]["interpretation"])

### What survives

Once you control for location and year, the elasticity of price with respect to area is
about **+0.25** — well below 1. A slope below 1 does mean unit price falls as size rises,
so the *direction* of the popular claim survives. What does not survive is its
**magnitude**: the raw −0.51 is mostly arithmetic.

This is a good example of why a placebo test is worth the effort. The original analysis
reported the raw correlation as a finding.

In [ ]:
from src.evaluation.hypothesis_tests import describe_bad_control

# H1's robustness section: what happens if we also control for the neighbourhood's
# recent price level?
print(describe_bad_control())

In [ ]:
robust = pd.read_csv(TABLES / "h1_robustness_neighbourhood.csv")
robust.head(4)

### A note on "bad controls"

`pc_med_price_12m` — the neighbourhood's recent price level — is a *bad control* here.

Why: area influences price, and price influences the neighbourhood median. So the
neighbourhood median sits **downstream** of the thing we are estimating. Conditioning on
it absorbs part of the very variation we want, biasing the area coefficient toward zero.

We still run it, because it is informative — if the estimate barely moves, we learn that
the area gradient does not run through neighbourhood price momentum. But it is reported
as a robustness check, never as the headline.

## 3. H3 — did the Metro opening raise nearby prices?

On **2019-05-26** the first stage of Sydney Metro opened, with 13 new stations. If the
market valued the new rail access, prices near those stations should have risen relative
to prices further away.

The method is a **difference-in-differences event study**:

* **Treated** postcodes: within 2 km of a station
* **Control** postcodes: more than 3 km away, in the same broad region
* Compare the *change* in treated prices with the *change* in control prices, month by
  month around the opening

### The assumption that must hold, and the test that decides everything

Difference-in-differences only identifies a causal effect if the two groups were on
**parallel trends before** the event. If treated postcodes were already rising faster,
the estimate is contaminated.

So we test the pre-period coefficients **jointly**: are they all zero? This test decides
whether we are allowed to say anything causal.

In [ ]:
from src.data.events import load_events, treatment_summary, assign_treatment, build_event_panel, EventStudySpec

ev = load_events()
print(ev[["event_id", "event_name", "event_date", "opened_stations"]].to_string(index=False))

In [ ]:
from src.causal.event_study import balance_panel
from src.data.geocode_features import build_postcode_geocode_distances
from src.data.events import _stations_at_event

spec = EventStudySpec()
transport_lookup = pd.read_csv("data/interim/postcode_transport_features.csv", dtype={"postcode": "string"})

# Distances are computed from individual station *entrances*, not postcode centroids,
# so a postcode is treated if any part of it falls inside the radius.
station_names = _stations_at_event(spec.event_id)
print(f"{len(station_names)} stations opened at {spec.event_id}: {station_names[:4]} ...")
geometry, geocode_report = build_postcode_geocode_distances(stations=station_names)
print(f"geocoded postcode distances: {len(geometry):,} postcodes")

treatment = assign_treatment(transport_lookup, spec, geocode_features=geometry)
summary = treatment_summary(treatment, radius_km=spec.radius_km)

print()
print("treated postcodes :", summary["treated_postcodes"])
print("control postcodes :", summary["control_postcodes"])
print("treated list      :", summary["treated_list"])

In [ ]:
# The balanced postcode x month panel is written by pipelines/06_event_study.py. It is
# produced from the full transaction file, so it is read here rather than rebuilt.
panel_path = TABLES / "event_study_panel.parquet"
balanced = pd.read_parquet(panel_path) if panel_path.exists() else None

if balanced is None:
    print("event_study_panel.parquet not found - run pipelines/06_event_study.py first.")
else:
    print(f"balanced panel: {len(balanced):,} postcode-months, columns: {list(balanced.columns)}")
    balanced.head()

In [ ]:
# The pre-trend test is the gate on any causal reading, so read it from the table the
# event study wrote (see the next cell for why we do not refit it here).
pretrend = pd.read_csv(TABLES / "event_study_pretrend.csv")
print("pre-trend coefficients (joint Wald test that all pre-period terms are zero):")
pretrend

In [ ]:
event_meta = json.loads((TABLES / "event_study_meta.json").read_text())
main = event_meta["main"]

print("PUBLISHED event-study result:")
print(f"  treated postcodes : {event_meta['treatment_summary']['treated_postcodes']}")
print(f"  controls          : {event_meta['treatment_summary']['control_postcodes']}")
print(f"  pre-trend p-value : {main['pretrend']['p_value']:.3g}")
print(f"  mean post-period coefficient: {main['mean_post_coef']:+.4f} log10 "
      f"(about {main['implied_pct_on_unit_price']:+.1%} on the median unit price)")
print()
print("A small p-value REJECTS the parallel-trends assumption.")
print("That is the number that decides whether a causal claim is allowed.")
print()
print("The regression itself is deliberately not refitted in this notebook: the balanced")
print("panel has ~1,600 postcode-months and the two-way fixed-effects estimator iterates to")
print("convergence, which is not worth repeating when pipelines/06_event_study.py has")
print("already recorded the coefficients in reports/tables/.")

In [ ]:
# Placebo: pretend the event happened 3 years earlier. If we "find" an effect at a date
# when nothing happened, the design is picking up pre-existing divergence.
placebo = pd.read_csv(TABLES / "event_study_placebo.csv")
dose = pd.read_csv(TABLES / "event_study_dose_response.csv")
print("placebo:")
print(placebo.to_string(index=False))
print()
print("dose-response:")
print(dose.to_string(index=False))

### The honest conclusion

**The pre-trend test rejects.** Treated and control postcodes were already on different
price paths before May 2019. That means the Metro coefficients **cannot be read as a
causal effect** — they are descriptive only, and this notebook reports them that way.

This is a negative result, and it is worth stating plainly rather than dressing up. It
is also why the distance-gradient (dose-response) specification matters: it uses
*continuous* distance rather than a treated/control split, so it is less dependent on the
parallel-trends assumption, and there the estimate is small and negative (a bigger
post-opening change closer to a station) — consistent with a rail premium, but weak.

## 4. Models — rolling-origin cross-validation

Random K-fold cross-validation is **wrong for time series**: it trains on 2020 to predict
2015. We use **rolling origin** (also called walk-forward) validation instead.

```
fold 0:  |---- train 2001-2007 ----|-emb-|-- valid 2008 --|
fold 1:  |------- train 2001-2009 -------|-emb-|-- valid 2010 --|
...
fold 7:  |------------- train 2001-2021 -------------|-emb-|-- valid 2022 --|
                                                    holdout 2023, untouched
```

Three details that matter:

* **Expanding window** — the training set always starts in 2001 and grows. This mirrors
  what you would actually do in production.
* **Embargo** — a one-month gap between train and validation, because contract dates to
  registration lag.
* **Purge by dwelling** — if the same dwelling sold in both the train and validation
  windows, the training copy is removed. Otherwise the model has already seen that house.

And a **final holdout**: the last 12 months are set aside and scored exactly once, after
all model choices are frozen. That is the only genuinely out-of-sample number.

In [ ]:
comparison = pd.read_csv(TABLES / "model_comparison.csv")
print("Pooled across the 8 validation folds, ranked by RMSLE:")
comparison[["model", "n", "rmsle", "r2_log10", "mdape_pct", "mae_aud"]]

In [ ]:
holdout = pd.read_csv(TABLES / "model_comparison_holdout.csv")
print("Final holdout — 2023, never used for any model choice:")
holdout[["model", "n", "rmsle", "r2_log10", "mdape_pct", "mae_aud", "is_best_from_cv"]]

In [ ]:
meta = json.loads((TABLES / "training_meta.json").read_text())
print("configuration actually used:")
for k in ("rows", "max_train_rows", "min_train_months", "horizon_months",
          "step_months", "embargo_months", "holdout_window", "best_model"):
    print(f"  {k:18s} {meta[k]}")
print()
print("max_train_rows = 0 means the window genuinely expands.")
print("With the default (400,000) it would silently become a ~4-year sliding window.")

In [ ]:
# Per-fold detail. Row counts must GROW across folds — that is the expanding window.
by_fold = pd.read_csv(TABLES / "model_scores_by_fold.csv")
pivot = by_fold.pivot_table(index="fold", columns="model", values="rmsle")
pivot["train_rows"] = by_fold.groupby("fold")["train_rows"].first()
pivot.round(4)

### How to read these three tables

**The benchmark is the point.** `naive_postcode_month` predicts every sale using its
postcode's trailing 12-month median price — one number, using only lagged information.
Any model that cannot beat it is not adding value.

The best model wins by roughly **12%** on RMSLE (about 0.318 vs 0.362). That is a
consistent but **modest** improvement. A single number — where you are, and roughly
when — gets you most of the way there.

**RMSLE vs MdAPE.** The two can disagree. RMSLE lives in log space, so it treats a 40%
miss on a cheap house and on an expensive one identically. MdAPE is a raw percentage on
the dollar scale, dominated by cheap properties. If a model wins on RMSLE but loses on
MdAPE, it is better on the expensive tail and slightly worse on typical cheap sales.

In [ ]:
# Where does each model actually win? Split the holdout by actual-price quintile.
band = pd.read_csv(TABLES / "model_scores_grouped.csv")
band[band.grouped_by == "area_band"][["group", "n", "rmsle", "mdape_pct", "mae_aud"]]

## 5. Which features earn their place?

The features are cumulative layers. Layer L1 adds calendar variables to a base of
property and location; L2 adds labels; L3 adds the lagged neighbourhood prices; and so
on. The ablation fits each layer and records the change in RMSLE.

**It is not enough to show a feature improves the score.** A large feature set will
improve almost anything by chance. So we also run two falsification tests:

* **Permutation placebo** — the same columns with values shuffled across postcodes. The
  distribution and missingness pattern survive, but the link to price is destroyed. If a
  model scores well on *that*, the gain was not real.
* **Time reversal** — forward-looking windows (`t+1..t+12`) instead of lagged ones. If the
  future version predicts *better*, the lag and embargo machinery is not doing its job.

In [ ]:
ablation = pd.read_csv(TABLES / "feature_ablation.csv")
ablation

In [ ]:
# The per-layer deltas, read straight from the table.
print("marginal contribution of each layer (negative = improves RMSLE):")
for _, r in ablation.dropna(subset=["rmsle_delta_vs_prev"]).iterrows():
    verdict = "improves" if r.rmsle_delta_vs_prev < 0 else "HURTS"
    print(f"  {r.layer:24s} {r.rmsle_delta_vs_prev:+.4f} RMSE  "
          f"{int(r.folds_improved_vs_prev)}/{int(r.folds)} folds  {verdict}")
print()
print("Calendar features contribute most. The lagged neighbourhood price statistics")
print("earn their place too. The liquidity/dispersion block does not.")

In [ ]:
placebo = pd.read_csv(TABLES / "feature_value_placebo.csv")
print("falsification tests (higher RMSLE = worse, as it should be):")
print(placebo.groupby("placebo")["rmsle"].agg(["mean", "min", "max", "count"]).round(4))

In [ ]:
import matplotlib.pyplot as plt

for name in ("fig5_rolling_cv.png", "fig6_residual_diagnostics.png", "fig9_dose_response.png"):
    p = FIGURES / name
    if p.exists():
        from IPython.display import Image, display
        display(Image(filename=str(p)))

## 6. The report

Everything above is assembled into a single document, with every figure regenerated and
every number read from the CSVs rather than typed by hand.

In [ ]:
report = (REPORTS_DIR / "final_report.md").read_text(encoding="utf-8")
print(f"final_report.md: {len(report):,} characters")
print()
# The headline findings, straight out of the document
start = report.index("## 0. Headline findings")
end = report.index("## 1.")
print(report[start:end])

## 7. How the tables this notebook read were produced

The commands below are for reference — you do not need them to read this notebook. Run
them from the repository root, after placing `nsw_property_data.csv` there.

```powershell
python pipelines/01_build_clean.py          # ~3 min   -> clean.parquet
python pipelines/02_build_features.py       # ~2 min   -> features.parquet
python pipelines/04_train_models.py --min-train-months 84 --step-months 24 `
       --holdout-months 12 --max-train-rows 0 --rf-trees 200 --rf-jobs 16 `
       --xgb-rounds 600 --xgb-device cuda   # ~43 min
python pipelines/05_hypothesis_tests.py     # ~2 min
python pipelines/06_event_study.py          # seconds
python pipelines/08_feature_value_test.py --device cuda   # ~6 min
python pipelines/07_make_report.py          # seconds
```

Or the whole chain in one process:

```powershell
python pipelines/run_chain.py --rf-jobs 16
```

### And check the tables before trusting them

```powershell
python pipelines/00_check_staleness.py
```

It reports which tables predate the feature frame **and** cross-checks them against each
other: row counts, whether every model has predictions, whether the grouped diagnostics
sum to the right total, and whether the report is quoting the current numbers. A stale
table is invisible otherwise — the report would show fresh prose next to old figures.

## Summary of the findings

1. **Bigger houses cost more, but not proportionally.** Pooled, area and price correlate
   *negatively* (−0.11) because big houses sit where land is cheap. Within a postcode and
   year the elasticity is **+0.25**.
2. **The "cheaper per sqm" claim is mostly arithmetic.** Of the raw −0.51 correlation,
   about **85%** survives shuffling prices at random — it is the ratio's denominator. The
   economic part is the +0.25 elasticity, which is below 1.
3. **The Metro study cannot support a causal claim.** The parallel-trends test rejects
   (p ≈ 1e-06), so treated postcodes were already diverging before 2019.
4. **The models beat the benchmark, modestly.** Best pooled RMSLE **0.318** against
   **0.362** for the postcode-median benchmark — about 12%.
5. **The engineered lagged features do earn their place**, but calendar features and the
   region labels contribute more.